# Aqua Quenchers benchmark protocol (unexecuted)
Prepared 6 November 2026. Read BENCHMARK_ANALYSIS.md and classifier/README.md first. This notebook is an optional local/Colab front end for the existing CLI, not benchmark results. No dependency installation, downloads, uploads, release packaging or network calls are performed here. Use an approved isolated offline environment with dependencies already installed. Colab requires independently approved private storage/runtime; it is not required.

**NOT RUN:** Owner-reviewed corpus, frozen release model, physical Owner tablet and device measurements are absent. Synthetic correctness is not station accuracy. Keep raw remarks, error-analysis files, logs and all outputs outside public GitHub/notebook commits. Do not run target code in an agent environment without its required complete OS sandbox. All execution switches default to False.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_ROOT = None   # Absolute path to the reviewed Aqua-Quenchers-BMS checkout.
DATASET = None        # Absolute path to private, Owner-reviewed, redacted JSONL.
EXPERIMENT = None     # Absolute NEW private experiment directory, outside the checkout.
TIMING_REPORT = None  # Absolute NEW private timing JSON path, outside the checkout.
MODEL_VERSION = None # Approved candidate identifier, e.g. EXP-LR-2026-v1.
SYNTHETIC_ONLY = False
RUN_PREPARE_AND_TRAIN = False
RUN_HELD_OUT_ONCE = False
RUN_FIVE_PROCESS_TIMING = False


## Private input and command guard
The real Owner review is an external decision; setting reviewed:true does not establish it. Run from the actual source checkout: current training provenance uses Git from its working directory. The wrapper below fixes cwd explicitly and requires a clean reviewed source commit. Never remove freeze/evaluation markers to get another favorable test result. Each command saves exclusive private logs rather than printing raw dataset-derived results into this notebook.


In [ ]:
def context():
    if any(value is None for value in (PROJECT_ROOT, DATASET, EXPERIMENT, MODEL_VERSION)):
        raise ValueError("Set the reviewed repository and private input/output paths first.")
    repo = Path(PROJECT_ROOT).expanduser().resolve(strict=True)
    dataset = Path(DATASET).expanduser().resolve(strict=True)
    experiment = Path(EXPERIMENT).expanduser().resolve()
    if not (repo / "classifier/aqua_classifier/__main__.py").is_file():
        raise ValueError("PROJECT_ROOT must be the actual Aqua Quenchers source checkout.")
    if not dataset.is_file() or dataset.stat().st_size > 32 * 1024 * 1024:
        raise ValueError("Dataset must be a private JSONL file no larger than 32MiB.")
    if dataset.is_relative_to(repo) or experiment.is_relative_to(repo):
        raise ValueError("Keep private datasets and experiments outside the source checkout.")
    if not isinstance(MODEL_VERSION, str) or not MODEL_VERSION.strip():
        raise ValueError("A reviewed model-version identifier is required.")
    return repo, dataset, experiment


def run_private(arguments, log_name, *, timing=False):
    repo, dataset, experiment = context()
    log_dir = experiment.parent / (experiment.name + "-private-logs")
    log_dir.mkdir(mode=0o700, parents=False, exist_ok=True)
    if log_dir.is_symlink() or log_dir.stat().st_mode & 0o077:
        raise ValueError("Private log directory must be restricted to its custodian.")
    for name in ("home", "tmp"):
        (log_dir / name).mkdir(mode=0o700, exist_ok=True)
    # Explicit child environment: no inherited credentials, proxy tokens or user config.
    child_env = {"PATH": str(Path(sys.executable).parent) + ":/usr/bin:/bin",
                 "HOME": str(log_dir / "home"), "TMPDIR": str(log_dir / "tmp"),
                 "PYTHONPATH": str(repo / "classifier"), "PYTHONNOUSERSITE": "1",
                 "PYTHONDONTWRITEBYTECODE": "1", "LANG": "C.UTF-8",
                 "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"}
    status = subprocess.run(["git", "status", "--porcelain", "--", "classifier",
                             "classifier-java", "app", "supabase"], cwd=repo,
                            env=child_env, check=True, text=True, capture_output=True)
    if status.stdout.strip():
        raise ValueError("Review and commit application/classifier changes before an experiment.")
    module = "aqua_classifier.benchmark" if timing else "aqua_classifier"
    argv = [sys.executable, "-B", "-m", module, *arguments]
    if SYNTHETIC_ONLY:
        argv.append("--synthetic")
    log_path = log_dir / log_name
    fd = os.open(log_path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(fd, "w", encoding="utf-8") as stream:
        completed = subprocess.run(argv, cwd=repo, env=child_env, stdout=stream,
                                   stderr=subprocess.STDOUT, timeout=1800, check=False)
    if completed.returncode:
        raise RuntimeError("Command failed; privately review its log and any freeze/evaluation marker before retrying.")
    print({"command": module, "evidence_kind": "SYNTHETIC ONLY" if SYNTHETIC_ONLY else "REVIEWED-CANDIDATE: approval/evaluation still separate",
           "completed": True, "results": "Saved only in configured private paths; inspect before sharing."})


## 1. Prepare/train on a new experiment
Grouped60/20/20 targets, seed42; train-only vocabulary/IDF, validation-only LR/lambda/NB/policy tuning. Record actual split/class counts and data/source hashes. No held-out evaluation in this step. Review output privately before enabling step2. Existing experiment directories are rejected by this notebook step.


In [ ]:
if RUN_PREPARE_AND_TRAIN:
    _, dataset, experiment = context()
    if experiment.exists():
        raise FileExistsError("Use a new experiment; preserve prior/partial evidence.")
    run_private(["prepare", "--dataset", str(dataset), "--output", str(experiment)], "01-prepare.log")
    run_private(["train", "--dataset", str(dataset), "--output", str(experiment),
                 "--model-version", MODEL_VERSION], "02-train.log")
else:
    print("NOT RUN: preparation/training require explicit private inputs and execution switch.")


## 2. Once-only frozen held-out evaluation
Enable only after reviewing and freezing split/model/policy/gates. A started evaluation is not a license to tune or rerun test labels. Both LR and weighted NB produce comparison metrics. Do not copy raw error-analysis remarks to notebook outputs or GitHub. Read private test-results.json, report actual supports/confusion matrices/macro metrics and failed gates honestly.


In [ ]:
if RUN_HELD_OUT_ONCE:
    _, dataset, experiment = context()
    run_private(["evaluate", "--dataset", str(dataset), "--output", str(experiment)], "03-held-out.log")
else:
    print("NOT RUN: once-only held-out evaluation has not been authorized by this switch.")


## 3. Training-process timing (separate from held-out accuracy)
One discarded fresh-process warmup plus five fresh prepare/train processes; median/range/peak RSS/environment. This measures preparation+training+tuning, not optimizer-only time. Use a NEW private timing-report path. It never evaluates the held-out set. Synthetic timing is infrastructure-only evidence.


In [ ]:
if RUN_FIVE_PROCESS_TIMING:
    repo, dataset, _ = context()
    if TIMING_REPORT is None:
        raise ValueError("Set a new private TIMING_REPORT path first.")
    timing_path = Path(TIMING_REPORT).expanduser().resolve()
    if timing_path.is_relative_to(repo) or timing_path.exists():
        raise ValueError("Timing report must be a new private path outside the checkout.")
    run_private(["--dataset", str(dataset), "--output", str(timing_path)], "04-training-timing.log", timing=True)
else:
    print("NOT RUN: five-process training timing requires explicit private inputs and switch.")


## 4. Actual Android/tablet measurement and acceptance (external)
Use ClassifierDeviceBenchmark with the approved model/digest and owner_tablet_id, separately from sign-in/business controllers.100warmups,1000timed calls: median/p95 latency, incremental Java heap including load, and separate Android process RSS/PSS. Record device/runtime/model/conditions and real-model Python/Java/Android parity.

| Gate fixed before final testing | Current state |
|---|---|
| Macro-F1 >=0.80; each category recall >=0.60 | NOT RUN: reviewed corpus/model missing |
| Approved model <=2MiB | NOT RUN: release model missing; loader/export cap exists |
| Owner-tablet p95 <=100ms | NOT RUN: actual tablet/model missing |
| Incremental memory <=32MiB | NOT RUN: actual tablet/heap/process evidence missing |
| Target Android parity | NOT RUN for approved real model |

No generated model is automatically release-approved. Explicit Owner category confirmation and backend save authorization remain required. Manual expense entry remains available without a release model.
